In [ ]:
from codecarbon import EmissionsTracker

tracker = EmissionsTracker()  
tracker.start()
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

[codecarbon WARNING @ 17:25:25] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 17:25:25] [setup] RAM Tracking...
[codecarbon INFO @ 17:25:25] [setup] CPU Tracking...
[codecarbon WARNING @ 17:25:27] We saw that you have a Intel(R) Core(TM) i9-14900K but we don't know it. Please contact us.
[codecarbon WARNING @ 17:25:27] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 17:25:27] CPU Model on constant consumption mode: Intel(R) Core(TM) i9-14900K
[codecarbon WARNING @ 17:25:27] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 17:25:27] [setup] GPU Tracking...
[codecarbon INFO @ 17:25:27] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 17:25:27] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from collections import defaultdict
from genomic_benchmarks.dataset_getters.pytorch_datasets import get_dataset
# Import the genomic benchmark dataset from Hugging Face
from genomic_benchmarks.dataset_getters.pytorch_datasets import DemoMouseEnhancers

# --------------------
# Data Preprocessing
# --------------------




    # Load the genomic benchmark dataset from Hugging Face
train_dset = get_dataset("human_enhancers_ensembl", split="train", version=0)
test_dset  = get_dataset("human_enhancers_ensembl", split="test", version=0)

    # Extract sequences and labels from the datasets
train_seqs = [item[0] for item in train_dset]
train_labels = [item[1] for item in train_dset]
test_seqs = [item[0] for item in test_dset]
test_labels = [item[1] for item in test_dset]
import pandas as pd
# converting the lists to train and test csv
train_df = pd.DataFrame({
    'sequence':train_seqs,
    'target': train_labels
})
test_df = pd.DataFrame({
    'sequence':test_seqs,
    'target': test_labels
})   



[codecarbon INFO @ 17:25:42] Energy consumed for RAM : 0.000083 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 17:25:42] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 17:25:42] Energy consumed for All CPU : 0.000177 kWh
[codecarbon INFO @ 17:25:42] Energy consumed for all GPUs : 0.001224 kWh. Total GPU Power : 293.4472500452032 W
[codecarbon INFO @ 17:25:42] 0.001485 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:25:57] Energy consumed for RAM : 0.000167 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 17:25:57] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 17:25:57] Energy consumed for All CPU : 0.000354 kWh
[codecarbon INFO @ 17:25:57] Energy consumed for all GPUs : 0.002442 kWh. Total GPU Power : 292.07087705052356 W
[codecarbon INFO @ 17:25:57] 0.002963 kWh of electricity used since the beginning.


In [5]:

test_df_new = pd.read_csv(r"D:\genomic benchmark\human_enhancers_ensembl\dnabert-2\ienhancertest.csv")

In [6]:
def kmer_tokenizer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

train_df["kmer"] = train_df["sequence"].apply(lambda x: kmer_tokenizer(x))
test_df["kmer"] = test_df["sequence"].apply(lambda x: kmer_tokenizer(x))

In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["kmer", "target"]])
test_dataset = Dataset.from_pandas(test_df[["kmer", "target"]])

In [8]:
from transformers import AutoTokenizer,  AutoModelForMaskedLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("LongSafari/hyenadna-large-1m-seqlen-hf", trust_remote_code=True)

# # Load model using EsmModel
# model = AutoModelForMaskedLM.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-100m-multi-species", trust_remote_code=True).to(device)


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
   "LongSafari/hyenadna-large-1m-seqlen-hf",
    num_labels=2,
    trust_remote_code=True
).to(device)


Some weights of HyenaDNAForSequenceClassification were not initialized from the model checkpoint at LongSafari/hyenadna-large-1m-seqlen-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
def tokenize_function(examples):
    return tokenizer(
        examples["kmer"],
        padding="max_length",       # pad all sequences to max_length
        truncation=True,
        max_length=512              # ensure fixed 512 tokens
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


# 6) Now rename "target" → "labels"
train_dataset = train_dataset.rename_column("target", "labels")
test_dataset  = test_dataset.rename_column("target", "labels")
train_dataset.set_format("torch", columns=["input_ids",  "labels"])
test_dataset.set_format("torch", columns=["input_ids",  "labels"])

[codecarbon INFO @ 17:26:12] Energy consumed for RAM : 0.000250 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 17:26:12] Delta energy consumed for CPU with constant : 0.000178 kWh, power : 42.5 W
[codecarbon INFO @ 17:26:12] Energy consumed for All CPU : 0.000532 kWh
[codecarbon INFO @ 17:26:12] Energy consumed for all GPUs : 0.003667 kWh. Total GPU Power : 292.3251382696722 W
[codecarbon INFO @ 17:26:12] 0.004449 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:26:27] Energy consumed for RAM : 0.000333 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 17:26:27] Delta energy consumed for CPU with constant : 0.000176 kWh, power : 42.5 W
[codecarbon INFO @ 17:26:27] Energy consumed for All CPU : 0.000708 kWh
[codecarbon INFO @ 17:26:27] Energy consumed for all GPUs : 0.004892 kWh. Total GPU Power : 294.9551251640241 W
[codecarbon INFO @ 17:26:27] 0.005934 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:26:42] Energy consumed for RAM : 0.000416 kWh. RAM Power : 2

In [11]:
# 6) Now rename "target" → "labels"

train_dataset.set_format("torch", columns=["input_ids",  "labels"])
test_dataset.set_format("torch", columns=["input_ids",  "labels"])

In [12]:
# train_dataset = train_dataset.rename_column("target", "labels")
# test_dataset  = test_dataset.rename_column("target", "labels")
# train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
# test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [13]:
from transformers import TrainingArguments

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir="D:\genomic benchmark\datasets\dnabert",

    disable_tqdm=False,
    # evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    # evaluation_strategy="no", 
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    report_to="none",
)

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
   
    compute_metrics=compute_metrics,
)


In [15]:
import time

# Start timer
start_time = time.time()

# Train the model
trainer.train()

# End timer
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Training took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
# tracker.stop()


Step,Training Loss
10,0.693600
20,0.674700
30,0.681200
40,0.669600
50,0.660100
60,0.663100
70,0.678500
80,0.655800
90,0.639700
100,0.684200


[codecarbon INFO @ 17:29:28] Energy consumed for RAM : 0.001321 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 17:29:28] Delta energy consumed for CPU with constant : 0.000174 kWh, power : 42.5 W
[codecarbon INFO @ 17:29:28] Energy consumed for All CPU : 0.002813 kWh
[codecarbon INFO @ 17:29:28] Energy consumed for all GPUs : 0.019761 kWh. Total GPU Power : 295.18952143564536 W
[codecarbon INFO @ 17:29:28] 0.023896 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:29:28] 0.069003 g.CO2eq/s mean an estimation of 2,176.08725366196 kg.CO2eq/year
[codecarbon INFO @ 17:29:43] Energy consumed for RAM : 0.001404 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 17:29:43] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 17:29:43] Energy consumed for All CPU : 0.002991 kWh
[codecarbon INFO @ 17:29:43] Energy consumed for all GPUs : 0.020981 kWh. Total GPU Power : 292.36251391175705 W
[codecarbon INFO @ 17:29:43] 0.025376 kWh of electricity used


🕒 Training took 3204.06 seconds (53.40 minutes)


[codecarbon INFO @ 18:22:44] Energy consumed for RAM : 0.019068 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 18:22:44] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 18:22:44] Energy consumed for All CPU : 0.040541 kWh
[codecarbon INFO @ 18:22:44] Energy consumed for all GPUs : 0.280177 kWh. Total GPU Power : 294.283901976108 W
[codecarbon INFO @ 18:22:44] 0.339787 kWh of electricity used since the beginning.
[codecarbon INFO @ 18:22:59] Energy consumed for RAM : 0.019152 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 18:22:59] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 18:22:59] Energy consumed for All CPU : 0.040718 kWh
[codecarbon INFO @ 18:22:59] Energy consumed for all GPUs : 0.281419 kWh. Total GPU Power : 297.9535139544012 W
[codecarbon INFO @ 18:22:59] 0.341289 kWh of electricity used since the beginning.
[codecarbon INFO @ 18:23:14] Energy consumed for RAM : 0.019235 kWh. RAM Power : 20

In [16]:
tracker.stop()

[codecarbon INFO @ 18:24:06] Energy consumed for RAM : 0.019520 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 18:24:06] Delta energy consumed for CPU with constant : 0.000074 kWh, power : 42.5 W
[codecarbon INFO @ 18:24:06] Energy consumed for All CPU : 0.041501 kWh
[codecarbon INFO @ 18:24:06] Energy consumed for all GPUs : 0.286926 kWh. Total GPU Power : 299.3317598856501 W
[codecarbon INFO @ 18:24:06] 0.347947 kWh of electricity used since the beginning.


0.24057427648306806

In [18]:
model.save_pretrained("D:\genomic benchmark\datasets\models and tokenizer/hyenadna-ffinetuned_humannontata", safe_serialization=False)


In [19]:
tokenizer.save_pretrained("D:\genomic benchmark\datasets\models and tokenizer/hyenadna-finetuned_humannontata", safe_serialization=False)


('D:\\genomic benchmark\\datasets\\models and tokenizer/hyenadna-finetuned_humannontata\\tokenizer_config.json',
 'D:\\genomic benchmark\\datasets\\models and tokenizer/hyenadna-finetuned_humannontata\\special_tokens_map.json',
 'D:\\genomic benchmark\\datasets\\models and tokenizer/hyenadna-finetuned_humannontata\\added_tokens.json')

In [20]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [22]:
test_df

,sequence,target,kmer
0,GCAATCCTCCAGCCTTGGCCTCCCAAAGTGCTGGGATTACAAGCAT...,0,GCAATC CAATCC AATCCT ATCCTC TCCTCC CCTCCA CTCC...
1,AAACTTTTCCTGCCAAAGTGTCTCCTACTGAAGTCTGTGGCTACTT...,0,AAACTT AACTTT ACTTTT CTTTTC TTTTCC TTTCCT TTCC...
2,ATCATAGGAATATGGTTTCTCTAATGCTATGTATGGCCCCCACCTC...,0,ATCATA TCATAG CATAGG ATAGGA TAGGAA AGGAAT GGAA...
3,CAACCCTTAAGTCCCTACCCTTAAGACCAGTGTAGTACAGATGGTC...,0,CAACCC AACCCT ACCCTT CCCTTA CCTTAA CTTAAG TTAA...
4,AACTGTCAACTCCTGTTTCTGATACTCTGAGGGGGACATGTCTCTC...,0,AACTGT ACTGTC CTGTCA TGTCAA GTCAAC TCAACT CAAC...
...,...,...,...
30965,TGCCACACACACGTGTGGGTGGAAAGAGAGTAAGATACTGAGACTG...,1,TGCCAC GCCACA CCACAC CACACA ACACAC CACACA ACAC...
30966,TGACTGAGGAAAGCCCGAGCTTTTTTTTTTTTTTTTTTGAGACGGA...,1,TGACTG GACTGA ACTGAG CTGAGG TGAGGA GAGGAA AGGA...
30967,GACAGAGTCTTGCTCTGTTGCCCAGGCTGTAGTGCAGTGGTGCCAT...,1,GACAGA ACAGAG CAGAGT AGAGTC GAGTCT AGTCTT GTCT...
30968,GCAAAAAAAAAAAAAAAAAAAAAATTAGAAAACAAAAAAATGCTTA...,1,GCAAAA CAAAAA AAAAAA AAAAAA AAAAAA AAAAAA AAAA...


In [23]:
import pandas as pd
test_df_im = pd.read_csv(r"D:\genomic benchmark\human_tata promoters\non_tata_vs_non_promoter_external_new.csv")


In [24]:
def kmer_tokenizer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])


test_df["kmer"] = test_df["sequence"].apply(lambda x: kmer_tokenizer(x))
test_df_im["kmer"] = test_df_im["sequence"].apply(lambda x: kmer_tokenizer(x))

In [27]:
test_df_new["kmer"] = test_df_new["sequence"].apply(lambda x: kmer_tokenizer(x))

In [25]:
test_dataset = Dataset.from_pandas(test_df[["kmer", "target"]])
test_dataset_im = Dataset.from_pandas(test_df_im[["kmer", "target"]])

In [28]:
test_dataset_new = Dataset.from_pandas(test_df_new[["kmer", "target"]])

In [26]:
def tokenize_function(examples):
    return tokenizer(
        examples["kmer"],
        padding="max_length",       # pad all sequences to max_length
        truncation=True,
        max_length=512              # ensure fixed 512 tokens
    )


test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset_im = test_dataset_im.map(tokenize_function, batched=True)

# 6) Now rename "target" → "labels"

test_dataset  = test_dataset.rename_column("target", "labels")
test_dataset_im  = test_dataset_new.rename_column("target", "labels")
test_dataset.set_format("torch", columns=["input_ids", "labels"])
test_dataset_im.set_format("torch", columns=["input_ids",  "labels"])

Map: 100%|██████████| 47542/47542 [00:54<00:00, 872.95 examples/s]


In [31]:
test_dataset_new = test_dataset_new.map(tokenize_function, batched=True)

Map: 100%|██████████| 400/400 [00:00<00:00, 1080.65 examples/s]


In [33]:

test_dataset_new.set_format("torch", columns=["input_ids",  "labels"])

In [34]:
from torch.utils.data import DataLoader

test_dataloader = DataLoader(
    test_dataset, batch_size=16, shuffle=False
)


In [36]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

all_preds = []
all_labels = []
i=0
import time

# Start timer
start_time = time.time()
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(model.device)
        # attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        i+=1
        if i%100==0:
            print(str(i)+" done")
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
# tracker.stop()


100 done
200 done
300 done
400 done
500 done
600 done
700 done
800 done
900 done
1000 done
1100 done
1200 done
1300 done
1400 done
1500 done
1600 done
1700 done
1800 done
1900 done

🕒 Evaluation took 93.49 seconds (1.56 minutes)


In [37]:
tracker.stop()

[codecarbon WARNING @ 18:38:54] Tracker already stopped !
[codecarbon WARNING @ 18:38:54] Background scheduler didn't run for a long period (888s), results might be inaccurate
[codecarbon INFO @ 18:38:54] Energy consumed for RAM : 0.024456 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 18:38:54] Delta energy consumed for CPU with constant : 0.010490 kWh, power : 42.5 W
[codecarbon INFO @ 18:38:54] Energy consumed for All CPU : 0.051991 kWh
[codecarbon INFO @ 18:38:54] Energy consumed for all GPUs : 0.360563 kWh. Total GPU Power : 298.3371062741914 W
[codecarbon INFO @ 18:38:54] 0.437011 kWh of electricity used since the beginning.


0.3021539317801313

In [38]:
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))


Accuracy:  0.7655
Precision: 0.7656
Recall:    0.7655
F1 Score:  0.7655

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.75      0.76     15485
           1       0.76      0.78      0.77     15485

    accuracy                           0.77     30970
   macro avg       0.77      0.77      0.77     30970
weighted avg       0.77      0.77      0.77     30970



In [39]:
import torch
torch.cuda.empty_cache()

In [40]:
from torch.utils.data import DataLoader

test_dataloader_new = DataLoader(
    test_dataset_new, batch_size=16, shuffle=False
)

In [42]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

all_preds = []
all_labels = []
i=0
import time

# Start timer
start_time = time.time()
with torch.no_grad():
    for batch in test_dataloader_new:
        input_ids = batch['input_ids'].to(model.device)
        # attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        i+=1
        if i%100==0:
            print(str(i)+" done")
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
# tracker.stop()



🕒 Evaluation took 1.22 seconds (0.02 minutes)


In [43]:
tracker.stop()

[codecarbon WARNING @ 18:41:15] Tracker already stopped !
[codecarbon WARNING @ 18:41:15] Background scheduler didn't run for a long period (140s), results might be inaccurate
[codecarbon INFO @ 18:41:15] Energy consumed for RAM : 0.025238 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 18:41:15] Delta energy consumed for CPU with constant : 0.001662 kWh, power : 42.5 W
[codecarbon INFO @ 18:41:15] Energy consumed for All CPU : 0.053653 kWh
[codecarbon INFO @ 18:41:15] Energy consumed for all GPUs : 0.372236 kWh. Total GPU Power : 298.47502394672165 W
[codecarbon INFO @ 18:41:15] 0.451127 kWh of electricity used since the beginning.


0.31191403711515986

In [44]:
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))


Accuracy:  0.5825
Precision: 0.5960
Recall:    0.5825
F1 Score:  0.5673

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.77      0.65       200
           1       0.63      0.40      0.49       200

    accuracy                           0.58       400
   macro avg       0.60      0.58      0.57       400
weighted avg       0.60      0.58      0.57       400



In [45]:
tracker.stop()

[codecarbon WARNING @ 18:42:10] Tracker already stopped !
[codecarbon WARNING @ 18:42:10] Background scheduler didn't run for a long period (55s), results might be inaccurate
[codecarbon INFO @ 18:42:10] Energy consumed for RAM : 0.025544 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 18:42:10] Delta energy consumed for CPU with constant : 0.000650 kWh, power : 42.5 W
[codecarbon INFO @ 18:42:10] Energy consumed for All CPU : 0.054303 kWh
[codecarbon INFO @ 18:42:10] Energy consumed for all GPUs : 0.376804 kWh. Total GPU Power : 298.44537755007724 W
[codecarbon INFO @ 18:42:10] 0.456651 kWh of electricity used since the beginning.


0.3157338251771273